<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/Gemini_Multi_Agent_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-generativeai

In [2]:
import os
from google.colab import userdata
import google.generativeai as genai

def configure():
  secret_name = "GOOGLE_API_KEY"
  api_key = userdata.get(secret_name)
  if not api_key:
    raise ValueError(f"Secret '{secret_name}' not found. Please check the name in Colab's Secrets.")

  genai.configure(api_key=api_key)
  return genai.GenerativeModel('gemini-1.5-pro-latest')

In [3]:
#Agent1: The Planner
def planner_agent(model, topic: str) -> list[str]:
    print("Planner Agent: Creating a research plan...")
    prompt = f"""
    You are an expert research planner. Your task is to break down the following topic
    into 3-5 specific, answerable questions. Return these questions as a Python list of strings.

    TOPIC: "{topic}"

    Example output: ["question 1", "question 2", "question 3"]
    """
    try:
        response = model.generate_content(prompt)
        plan_str = response.text.strip().replace('[', '').replace(']', '').replace('"', '')
        plan = [q.strip() for q in plan_str.split(',') if q.strip()]

        print("Plan created:")
        for i, q in enumerate(plan, 1):
            print(f"   {i}. {q}")
        return plan
    except Exception as e:
        print(f"Error in Planner Agent: {e}")
        return []

In [4]:
#Agent2: The Search Agent
def search_agent(model, question: str) -> str:
    print(f"Search Agent: Researching question: '{question}'...")
    try:
        search_tool = genai.protos.Tool(
            google_search_retrieval=genai.protos.GoogleSearchRetrieval(disable_attribution=True)
        )
        prompt = f"Provide a detailed answer to the following question: {question}"
        response = model.generate_content(prompt, tools=[search_tool])

        print("   - Information found.")
        return response.text
    except Exception as e:
        print(f"Error in Search Agent: {e}")
        return ""

In [5]:
#Agent3: The Synthesizer
def synthesizer_agent(model, topic: str, research_results: list) -> str:
    print("Synthesizer Agent: Writing the final report...")

    research_notes = ""
    for question, data in research_results:
        research_notes += f"### Question: {question}\n### Research Data:\n{data}\n\n---\n\n"

    prompt = f"""
    You are an expert research analyst. Your task is to synthesize the provided research notes
    into a comprehensive, well-structured report on the topic: "{topic}".

    The report should have an introduction, a body that covers the key findings from the notes,
    and a conclusion. Use the information from the research notes ONLY.

    ## Research Notes ##
    {research_notes}
    """
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error in Synthesizer Agent: {e}")
        return "Error: Could not generate the final report."

In [8]:

def main():
    try:
        model = configure()
    except ValueError as e:
        print(e)
        return

    print("\nHello! I am your AI Research Assistant.")
    topic = input("What topic would you like me to research today? ")

    if not topic.strip():
        print("A topic is required to begin research. Exiting.")
        return

    print(f"\nStarting research process for: '{topic}'")

    research_plan = planner_agent(model, topic)
    if not research_plan:
        print("Could not create a research plan. Exiting.")
        return

    research_results = []
    for question in research_plan:
        research_data = search_agent(model, question)
        if research_data:
            research_results.append((question, research_data))

    if not research_results:
        print("Could not find any information during research. Exiting.")
        return

    final_report = synthesizer_agent(model, topic, research_results)

    print("\n\n--- FINAL RESEARCH REPORT ---")
    print(f"## Topic: {topic}\n")
    print(final_report)
    print("--- END OF REPORT ---")

main()


Hello! I am your AI Research Assistant.
What topic would you like me to research today? The Death of Charlie Kirk

Starting research process for: 'The Death of Charlie Kirk'
Planner Agent: Creating a research plan...
Error in Planner Agent: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0
Please retry in 59.607072542s.
Could not create a research plan. Exiting.
